In [ ]:
# =========================================================================
# LTX-Video (Lightricks 2B DiT) 圖生影換裝視頻生成 Worker (首幀合格母圖對齊版)
# =========================================================================
import os, sys, io, time, gc, json, urllib.request
import torch, psutil
from PIL import Image

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

print("📦 正在安裝與確認 LTX-Video 視頻生成專用依賴庫...")
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers>=0.41.0", "transformers>=5.17.0", "accelerate", "imageio[ffmpeg]"], check=False)

from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

model_id = "Lightricks/LTX-Video"
print(f"🎬 [新世代 2B DiT 頂級寫實視頻模型] 正在載入: {model_id}...")

# 啟用 bfloat16 精度載入，僅需 ~4GB 顯存，完美適應 16GB GPU
pipe = LTXImageToVideoPipeline.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
)
pipe.to("cuda")

# 載入通過 7 項品質檢驗的首幀 2K 母圖
mother_url = "https://raw.githubusercontent.com/Chenponpon/colab-ai-media-lab/main/assets/mother_images/portrait_2.jpg"
print(f"📥 正在從 GitHub 載入母圖: {mother_url} ...")
urllib.request.urlretrieve(mother_url, "input_mother.jpg")
input_img = Image.open("input_mother.jpg").convert("RGB")
orig_w, orig_h = input_img.size
print(f"   首幀原始解析度: {orig_w}x{orig_h}")

# 鎖定 LTX-Video 直式最佳規格 (576x736，寬高均能被 32 整除，長寬比 0.7826 完美吻合母圖 0.7778，絕不拉伸)
vid_w, vid_h = 576, 736
input_img = input_img.resize((vid_w, vid_h), Image.LANCZOS)
print(f"   縮放至視頻標準解析度: {vid_w}x{vid_h}")

motion_prompt = "Three young Eurasian women wearing sheer tulle mesh and floral lace lingerie sets in front of a large mirror, gently pulling up and fitting delicate lace straps over their shoulders as they finish dressing, brushing long wavy hair back with their fingers, turning their torsos naturally to admire their hourglass curves in the mirror, realistic body movement and soft eye blinks."
negative_prompt = "nude, nudity, naked, topless, exposed nipples, areola, static image, worst quality, inconsistent motion, blurry, jittery, distorted, morphing limbs"

# 幀數必須滿足 8k + 1 規則：k=15 => 121 幀 (121 / 24fps = 5.04 秒電影級動態)
total_frames = 121
print(f"⏳ 正在啟動 LTX-Video 圖生影管線 ({total_frames} 幀 / ~5 秒 24fps 高階合規換裝動態)...")
t0 = time.time()
video_frames = pipe(
    image=input_img,
    prompt=motion_prompt,
    negative_prompt=negative_prompt,
    width=vid_w,
    height=vid_h,
    num_frames=total_frames,
    num_inference_steps=40,
    guidance_scale=3.5
).frames[0]

dt = round(time.time() - t0, 2)
print(f"✨ 視頻生成完畢！耗時: {dt} 秒，正在輸出 MP4...")

out_vid_name = "changing_clothes_video.mp4"
export_to_video(video_frames, out_vid_name, fps=24)
vid_sz_mb = round(os.path.getsize(out_vid_name) / (1024 * 1024), 2)
print(f"🎉 影片已成功輸出為 {out_vid_name} (體積: {vid_sz_mb} MB)！")

bench = {
    "engine": "Lightricks/LTX-Video (2B DiT Image-to-Video)",
    "gpu": gpu_name,
    "elapsed_sec": dt,
    "frames": total_frames,
    "fps": 24,
    "resolution": f"{vid_w}x{vid_h}",
    "video_file": out_vid_name,
    "file_size_mb": vid_sz_mb
}
with open("video_benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench, bf, indent=2)
